# 01 — Qwen3-8B public development + validation campaign

Run the full public campaign — Stage B (development, 42 scenarios, ≥ 3 seeds) and
Stage C (validation, 18 scenarios, 5 seeds preferred / 3 the floor) — over the
frozen gateway, with the paired allow-all and full-AegisGraph conditions. Start a
new campaign, or **resume** an interrupted one; score and bundle every run.

Every evaluation step calls `scripts/bench_campaign.py`, `scripts/bench_score.py`
and `scripts/bench_validate.py`; no campaign or scoring logic lives in a cell. The
campaign CLI never manages processes, so this notebook owns the gateway and the
policy publication.

**No real-model result exists in this repository.** This notebook commits nothing;
the run directories and bundles stay in the cloud (optionally checkpointed to
Google Drive).

Run this notebook from a checkout of the repository (see `notebooks/README.md`); it locates the repository root automatically and `chdir`s into it. Nothing is
committed: the run directories and bundles stay in the cloud.


In [ ]:
import hashlib, json, os, re, subprocess, sys, tempfile, time, urllib.request, zipfile
from pathlib import Path


def _repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "bench_campaign.py").is_file() or (
            candidate / "scripts" / "bench_validate.py"
        ).is_file():
            return candidate
    return start


configured_repo = os.environ.get("AEGISGRAPH_REPO")
REPO = (Path(configured_repo).expanduser() if configured_repo else _repo_root(Path.cwd())).resolve()
if not (REPO / "scripts" / "bench_campaign.py").is_file():
    cloud_root = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working")
    if not cloud_root.is_dir():
        cloud_root = Path.cwd()
    REPO = cloud_root / "IndabaX26"
    if REPO.exists():
        raise RuntimeError("cloud checkout exists but is not AegisGraph: %s" % REPO)
    repo_ref = os.environ.get("AEGISGRAPH_REF", "main")
    subprocess.run(["git", "clone", "--branch", repo_ref, "--depth", "1",
                    "https://github.com/OussemaHarrabi/IndabaX26.git", str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

SCRATCH = Path(os.environ.get("AEGISGRAPH_SCRATCH") or tempfile.mkdtemp(prefix="aegisgraph-notebook-"))
SCRATCH.mkdir(parents=True, exist_ok=True)
ISSUER_DIR = SCRATCH / "dev-issuer"
POLICY_ADMIN_TOKEN = SCRATCH / "policy-admin.jwt"
DECISION_TOKEN = SCRATCH / "decision.jwt"
GATEWAY = {"proc": None, "url": None, "ablation": None}


def sh(args, *, check=True, capture=False, env=None):
    """Run a command from the repository root and echo it.

    ``capture=True`` returns the completed process with its output printed, so a
    notebook cell never silently swallows a CLI's diagnostics.
    """
    printable = [str(item) for item in args]
    print("$", " ".join(printable))
    proc = subprocess.run(printable, cwd=str(REPO), text=True, capture_output=capture, env=env)
    if capture:
        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError("command failed (exit %d): %s" % (proc.returncode, " ".join(printable)))
    return proc


def run_dirs_from(output):
    """The run directories a campaign invocation printed."""
    return [Path(match) for match in re.findall(r"run directory:\s*(\S+)", output)]


def init_issuer():
    if not (ISSUER_DIR / "jwks.json").is_file():
        sh([sys.executable, "scripts/dev_issuer.py", "init", "--directory", ISSUER_DIR], capture=True)
    for subject, role, path in (
        ("bench-policy-admin", "policy_admin", POLICY_ADMIN_TOKEN),
        ("bench-decision-client", "decision_client", DECISION_TOKEN),
    ):
        minted = subprocess.run(
            [sys.executable, "scripts/dev_issuer.py", "mint", "--directory", ISSUER_DIR,
             "--tenant", "bench", "--subject", subject, "--role", role, "--ttl", "43200"],
            cwd=str(REPO), text=True, capture_output=True, check=True,
        )
        path.write_text(json.loads(minted.stdout)["token"], encoding="utf-8")
    print("issuer ready:", ISSUER_DIR)


def stop_gateway():
    proc = GATEWAY.get("proc")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except Exception:
            proc.kill()
    GATEWAY.update(proc=None, url=None, ablation=None)


def start_gateway(port=8091, ablation=None):
    """Start the frozen gateway on loopback; optionally as an ablated build.

    The campaign CLI never manages processes and never sets AEGISGRAPH_*, so the
    notebook owns the gateway lifecycle. An ablation is a process-level switch.
    """
    stop_gateway()
    env = dict(os.environ)
    env.update({
        "PYTHONPATH": "backend",
        "AEGISGRAPH_AUTH_MODE": "required",
        "AEGISGRAPH_JWT_ISSUER": "https://dev-issuer.aegisgraph.local",
        "AEGISGRAPH_JWT_AUDIENCE": "aegisgraph",
        "AEGISGRAPH_JWKS": str(ISSUER_DIR / "jwks.json"),
    })
    if ablation:
        env["AEGISGRAPH_ABLATION"] = ablation
    log_path = SCRATCH / ("gateway-%s.log" % (ablation or "defence"))
    log = log_path.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "aegisgraph.app:app",
         "--host", "127.0.0.1", "--port", str(port)],
        cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    url = "http://127.0.0.1:%d" % port
    GATEWAY.update(proc=proc, url=url, ablation=ablation)
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError("gateway exited early; see %s" % log_path)
        try:
            with urllib.request.urlopen(url + "/healthz", timeout=2) as response:
                if json.load(response).get("status") == "ok":
                    print("gateway ready at", url, "(ablation: %s)" % (ablation or "none"))
                    return url
        except Exception:
            time.sleep(0.5)
    raise TimeoutError("gateway did not become healthy; see %s" % log_path)


def publish_policies(url):
    sh([sys.executable, "scripts/bench_policies.py", "publish",
        "--defense-url", url, "--token-file", POLICY_ADMIN_TOKEN], capture=True)


def adapter_json(**overrides):
    """The frozen QwenConfig as the driver's inline --adapter-json value.

    The CLI takes the JSON itself (``--adapter-json '{"revision": ...}'``), not a
    path; a copy is also written to the scratch directory for the record.
    """
    from benchmark.qwen import QwenConfig
    payload = json.dumps(QwenConfig(**overrides).to_json(), sort_keys=True)
    (SCRATCH / "adapter.json").write_text(payload + "\n", encoding="utf-8")
    print(payload)
    return payload


def campaign(stage, condition, seeds, *, defense_url=None, adapter_json=None,
             resume=False, bundle=False, max_scenarios=None, dry_run=False):
    """Invoke the campaign CLI. The CLI owns all evaluation logic."""
    args = [sys.executable, "scripts/bench_campaign.py", "--stage", stage,
            "--condition", condition, "--model", MODEL, "--runs-dir", str(RUNS_DIR)]
    for seed in seeds:
        args += ["--seed", str(seed)]
    if defense_url:
        args += ["--defense-url", defense_url]
    if dry_run:
        args += ["--dry-run"]
    else:
        if MODEL == "qwen" and adapter_json is not None:
            args += ["--adapter-json", str(adapter_json)]
        if DECISION_TOKEN.is_file():
            args += ["--auth-token-file", str(DECISION_TOKEN)]
        if resume:
            args += ["--resume"]
        if max_scenarios is not None:
            args += ["--max-scenarios", str(max_scenarios)]
        if bundle:
            args += ["--bundle"]
    return sh(args, capture=True)


def score_run(run_dir):
    """Score a run directory into score.json/score.txt (protocol section 8)."""
    run_dir = Path(run_dir)
    table = sh([sys.executable, "scripts/bench_score.py", "--run", run_dir], capture=True).stdout
    (run_dir / "score.txt").write_text(table, encoding="utf-8")
    sh([sys.executable, "scripts/bench_score.py", "--run", run_dir, "--json",
        "--out", str(run_dir / "score.json")], capture=True)
    print("scored:", run_dir.name)


def verify_zip(zip_path):
    zip_path = Path(zip_path)
    digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    with zipfile.ZipFile(zip_path) as archive:
        corrupt = archive.testzip()
    print("bundle:", zip_path.name, "sha256:", digest, "integrity:", "ok" if corrupt is None else corrupt)
    return digest


In [ ]:
# @title Parameters
STAGES = ["B", "C"]

# Seeds are PREREGISTERED: the freeze block fixes the anchor (1729) and the full
# list is closed in the Stage A record before Stage B starts. [1729] alone is
# deliberately below the Stage B/C floor of 3, so the dry run below refuses it
# until the closed list is entered here. Do not pick seeds after seeing outcomes.
SEEDS_B = [1729, 2741, 3253, 4253, 5527]
SEEDS_C = [1729, 2741, 3253, 4253, 5527]

MODEL = "qwen"
RUNS_DIR = Path(os.environ.get("AEGISGRAPH_RUNS_DIR", str(SCRATCH / "runs")))
GATEWAY_PORT = 8091
DEVICE = "cuda"
RESUME = False          # True: continue the newest incomplete run dir per seed
MAX_SCENARIOS = None    # int: clean stop after N new scenarios per seed (resumable)
ENABLE_DRIVE = False    # True on Colab: checkpoint each run dir + bundle to Drive
DRIVE_DIR = "aegisgraph-qwen-campaign"

MODEL_REVISION = os.environ.get("AEGISGRAPH_MODEL_REVISION", "unpinned")
QUANTIZATION = "4bit"
DTYPE = "bfloat16"
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NEW_TOKENS = 768
THINKING = False
FORMAT_RETRIES = 1
SEEDS_BY_STAGE = {"B": SEEDS_B, "C": SEEDS_C}
print("stages:", STAGES, "seeds:", {stage: SEEDS_BY_STAGE[stage] for stage in STAGES})
print("runs directory:", RUNS_DIR)


## 1. Runtime, toolchain and model

In [ ]:
GPU_NAME, VRAM_GB = None, None
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        GPU_NAME, VRAM_GB = props.name, props.total_memory / 1024 ** 3
        print("GPU:", GPU_NAME, "| VRAM (GiB):", round(VRAM_GB, 1))
except Exception as error:
    print("torch is not importable here:", error)
ENV_OK = GPU_NAME is not None and VRAM_GB is not None and VRAM_GB >= 14.0
print("environment suitable:", ENV_OK)
if not ENV_OK:
    raise RuntimeError("provision a Colab/Kaggle GPU runtime (a free T4 is 15 GiB) before the campaign.")

import shlex
from benchmark.qwen import INSTALL_COMMAND, QwenConfig, QwenModelAdapter
command = shlex.split(INSTALL_COMMAND)
command[0] = sys.executable
sh(command, check=False)
adapter = QwenModelAdapter(QwenConfig(
    revision=MODEL_REVISION, quantization=QUANTIZATION, dtype=DTYPE,
    temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    thinking=THINKING, device=DEVICE, format_retries=FORMAT_RETRIES,
))
adapter.load()
resolved_identity = adapter.identity().to_json()["parameters"]
MODEL_REVISION = resolved_identity.get("resolved_revision") or MODEL_REVISION
if MODEL_REVISION == "unpinned":
    raise RuntimeError("the exact Hugging Face model revision could not be resolved")
if not resolved_identity.get("chat_template_sha256"):
    raise RuntimeError("the tokenizer chat-template digest could not be resolved")
if not resolved_identity.get("max_context_tokens"):
    raise RuntimeError("the model context limit could not be resolved")
print(json.dumps(resolved_identity, indent=2, sort_keys=True))
import gc
del adapter
gc.collect()
torch.cuda.empty_cache()


## 2. Preconditions and gateway

In [ ]:
validated = sh([sys.executable, "scripts/bench_validate.py"], capture=True).stdout
if "RESULT: PASS" not in validated:
    raise RuntimeError("the dataset did not validate; the campaign must not start")
seal = sh([sys.executable, "scripts/bench_seal.py", "verify"], capture=True).stdout
if '"ciphertext_matches": true' not in seal:
    raise RuntimeError("the seal did not verify; the campaign must not start")

init_issuer()
GATEWAY_URL = start_gateway(GATEWAY_PORT)
publish_policies(GATEWAY_URL)
ADAPTER_JSON = adapter_json(
    revision=MODEL_REVISION, quantization=QUANTIZATION, dtype=DTYPE,
    temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    thinking=THINKING, device=DEVICE, format_retries=FORMAT_RETRIES,
)


## 3. Dry-run the plan for every stage

Each dry run validates the plan (including the seed floor) without a socket, a
directory or a model. A refusal means the configuration is not closed yet.


In [ ]:
for stage in STAGES:
    for condition in ["defence", "control"]:
        check = campaign(stage, condition, SEEDS_BY_STAGE[stage],
                         defense_url=GATEWAY_URL, dry_run=True)
        if "RESULT: DRY-RUN" not in check.stdout:
            raise RuntimeError("the plan for %s/%s was refused; close the configuration" % (stage, condition))


## 4. Run the campaign (new or resume)

One invocation per (stage, condition) with the full seed list; the driver writes
one run directory per seed. `--resume` continues the newest incomplete directory
for each seed; a completed directory is never written to again. Every run dir is
recorded for the scoring and bundling steps below.


In [ ]:
ALL_RUNS = []
for stage in STAGES:
    for condition in ["defence", "control"]:
        completed = campaign(stage, condition, SEEDS_BY_STAGE[stage], defense_url=GATEWAY_URL,
                             adapter_json=ADAPTER_JSON, resume=RESUME, max_scenarios=MAX_SCENARIOS)
        produced = run_dirs_from(completed.stdout)
        ALL_RUNS += produced
        print(stage, condition, "->", [p.name for p in produced])
print("total run directories:", len(ALL_RUNS))


## 5. Score and bundle

The driver never scores and never bundles on its own: scoring is `bench_score.py`,
and `--bundle` is a standalone mode that re-finds the newest completed run dir for
each (model, stage, condition, seed), extends `hashes.sha256` to cover the score
files and writes a `.zip`.


In [ ]:
for run_dir in ALL_RUNS:
    if Path(run_dir).is_dir():
        score_run(run_dir)


In [ ]:
BUNDLES = []
needs_resume = RESUME or MAX_SCENARIOS is not None
for stage in STAGES:
    for condition in ["defence", "control"]:
        bundled = campaign(stage, condition, SEEDS_BY_STAGE[stage], defense_url=GATEWAY_URL,
                           resume=needs_resume, bundle=True)
        if "RESULT: BUNDLE OK" not in bundled.stdout:
            raise RuntimeError("bundling failed for %s/%s" % (stage, condition))
        for zip_path in re.findall(r"zip:\s*(\S+)", bundled.stdout):
            BUNDLES.append(Path(zip_path))
            verify_zip(Path(zip_path))
print("bundles:", [p.name for p in BUNDLES])


## 6. Google Drive checkpoints (optional)

Preserve the artifacts before the session ends. The checkpoint copy is a plain
copy; the repository is never modified.


In [ ]:
if ENABLE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive") / DRIVE_DIR
    destination.mkdir(parents=True, exist_ok=True)
    for run_dir in [Path(p) for p in ALL_RUNS if Path(p).is_dir()]:
        target = destination / run_dir.name
        if not target.exists():
            shutil.copytree(run_dir, target)
    for zip_path in BUNDLES:
        target = destination / zip_path.name
        if not target.exists():
            shutil.copy2(zip_path, target)
    print("checkpointed to", destination)
else:
    print("Drive checkpoints are disabled (ENABLE_DRIVE = False).")


## 7. Download the bundles

Download the `.zip` bundles before the session expires; verify their SHA-256
locally (`hashes.sha256` inside each run dir is the authoritative manifest).


In [ ]:
for zip_path in sorted(RUNS_DIR.glob("*.zip")):
    print("bundle:", zip_path, "sha256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())
print()
print("In Colab, download with:")
print("  from google.colab import files")
print("  files.download('<the bundle path above>')")
stop_gateway()
